# Lesson 6b: Manifold Learning Practical — t-SNE and UMAP

## Overview
Apply t-SNE and UMAP to real datasets. Learn parameter tuning, compare visualization results, and understand when manifold learning reveals true structure versus creating artifacts.

**Learning Goals:**
- Use scikit-learn t-SNE API
- Tune perplexity and understand its effects
- Use UMAP-learn library
- Compare parameter choices across datasets
- Recognize when visualization creates false clusters

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits, load_iris
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# Try to import UMAP (install with: pip install umap-learn)
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not installed. Run: pip install umap-learn")

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Part 1: t-SNE Perplexity Tuning

Perplexity roughly sets the effective number of neighbors. Too low → local noise. Too high → oversmooting global structure.

In [ ]:
# Load iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Standardize
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply t-SNE with different perplexities
perplexities = [5, 30, 50]
results_tsne = []

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, perplexity in enumerate(perplexities):
    print(f"Computing t-SNE with perplexity={perplexity}...")
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
    X_tsne = tsne.fit_transform(X_iris_scaled)
    results_tsne.append(X_tsne)
    
    # Plot
    ax = axes[idx]
    for target in np.unique(y_iris):
        ax.scatter(X_tsne[y_iris == target, 0], X_tsne[y_iris == target, 1],
                  label=iris.target_names[target], alpha=0.6)
    ax.set_title(f't-SNE (perplexity={perplexity})')
    ax.set_xlabel('t-SNE1')
    ax.set_ylabel('t-SNE2')
    ax.legend()

plt.tight_layout()
plt.show()

print("Observation: Lower perplexity emphasizes local structure (noise).")
print("             Higher perplexity better captures global clusters.")

## Part 2: UMAP Practical Workflow

In [ ]:
if HAS_UMAP:
    # Apply UMAP with different n_neighbors
    n_neighbors_list = [5, 15, 30]
    results_umap = []
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, n_neighbors in enumerate(n_neighbors_list):
        print(f"Computing UMAP with n_neighbors={n_neighbors}...")
        umap_model = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, random_state=42)
        X_umap = umap_model.fit_transform(X_iris_scaled)
        results_umap.append(X_umap)
        
        ax = axes[idx]
        for target in np.unique(y_iris):
            ax.scatter(X_umap[y_iris == target, 0], X_umap[y_iris == target, 1],
                      label=iris.target_names[target], alpha=0.6)
        ax.set_title(f'UMAP (n_neighbors={n_neighbors})')
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("Observation: n_neighbors controls global vs local structure balance.")
else:
    print("UMAP not available. Install with: pip install umap-learn")

## Part 3: t-SNE vs UMAP Comparison

In [ ]:
# Load larger dataset (digits)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Standardize
X_digits_scaled = StandardScaler().fit_transform(X_digits)

# Time both methods
print("Comparing speed (digits dataset, n=1797, d=64)...\n")

# t-SNE
start = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_digits = tsne.fit_transform(X_digits_scaled)
tsne_time = time.time() - start
print(f"t-SNE time: {tsne_time:.2f}s")

# UMAP
if HAS_UMAP:
    start = time.time()
    umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap_digits = umap_model.fit_transform(X_digits_scaled)
    umap_time = time.time() - start
    print(f"UMAP time: {umap_time:.2f}s")
    print(f"\nUMAP is {tsne_time/umap_time:.1f}x faster than t-SNE")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    ax = axes[0]
    scatter = ax.scatter(X_tsne_digits[:, 0], X_tsne_digits[:, 1],
                        c=y_digits, cmap='tab10', alpha=0.6, s=30)
    ax.set_title('t-SNE')
    plt.colorbar(scatter, ax=ax)
    
    ax = axes[1]
    scatter = ax.scatter(X_umap_digits[:, 0], X_umap_digits[:, 1],
                        c=y_digits, cmap='tab10', alpha=0.6, s=30)
    ax.set_title('UMAP')
    plt.colorbar(scatter, ax=ax)
    
    plt.tight_layout()
    plt.show()

## Part 4: Detecting Visualization Artifacts

Visualizations can deceive. Always verify with multiple methods.

In [ ]:
# Generate random data with no true structure
np.random.seed(42)
X_random = np.random.randn(100, 50)

# Apply manifold learning
tsne_random = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_random = tsne_random.fit_transform(X_random)

if HAS_UMAP:
    umap_random = umap.UMAP(n_neighbors=15, random_state=42)
    X_umap_random = umap_random.fit_transform(X_random)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].scatter(X_tsne_random[:, 0], X_tsne_random[:, 1], alpha=0.6)
    axes[0].set_title('t-SNE of Random Data (FAKE CLUSTERS!)')
    axes[0].set_xlabel('t-SNE1')
    axes[0].set_ylabel('t-SNE2')
    
    axes[1].scatter(X_umap_random[:, 0], X_umap_random[:, 1], alpha=0.6)
    axes[1].set_title('UMAP of Random Data (fewer artifacts)')
    axes[1].set_xlabel('UMAP1')
    axes[1].set_ylabel('UMAP2')
    
    plt.tight_layout()
    plt.show()
    
    print("⚠️  Both methods create structure in random data!")
    print("✓ UMAP shows fewer false clusters than t-SNE")
    print("✓ Always verify with domain knowledge, not just visualization")

## Summary

### Key Takeaways
1. **Perplexity/n_neighbors**: Balance local and global structure
2. **Speed**: UMAP significantly faster than t-SNE
3. **Artifacts**: Both methods create false clusters in random data
4. **Best practice**: Use both methods, verify with domain knowledge

### Practical Workflow
1. Standardize features
2. Try both t-SNE and UMAP
3. Vary perplexity/n_neighbors
4. Verify results match domain expectations
5. Use as exploratory tool, not final word on structure

### Next Steps
- TASK-UL15: Anomaly Detection (Isolation Forest, LOF)
- Advanced: Parametric t-SNE, UMAP uncertainty estimates